# Векторизация, Numba

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. "Оптимизация выполнения кода, векторизация, Numba"
* https://numba.pydata.org/numba-doc/latest/user/5minguide.html
* https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types
* https://numpy.org/doc/stable/reference/generated/numpy.vectorize.html
* https://numba.pydata.org/numba-doc/latest/user/vectorize.html


## Задачи для совместного разбора

In [2]:
import pandas as pd
import numpy as np
import numba
import string
import json

1. Сгенерируйте массив `A` из `N=1млн` случайных целых чисел на отрезке от 0 до 1000. Пусть `B[i] = A[i] + 100`. Посчитайте среднее значение массива `B`. Ускорьте вычисления при помощи numba

In [13]:
A = np.random.randint(0, 1000, size=(1000000,))

In [3]:
def f1(A):
    acc, cnt = 0, 0
    for x in A:
        acc += x + 100
        cnt += 1
    return acc / cnt

In [4]:
%%time
f1(A)

Wall time: 305 ms


599.601713

In [21]:
@numba.njit
def f2(A):
    s = pd.DataFrame()
    acc, cnt = 0, 0
    for x in A:
        acc += x + 100
        cnt += 1
    return acc / cnt

In [22]:
@numba.jit
def f3(A):
    s = pd.DataFrame()
    acc, cnt = 0, 0
    for x in A:
        acc += x + 100
        cnt += 1
    return acc / cnt

In [19]:
%%time
f3(A)

Wall time: 0 ns


599.268007

2. Напишите функцию, которая возвращает сумму всех чисел от 0 до x-1. Создайте массив, заполненный случайными целыми неотрицательными числами и примените функцию к каждому элементу массива.

In [5]:
def sum_x(x):
    return sum(range(x))

In [28]:
A = np.random.randint(0, 100, size=1_000_000)

In [7]:
%%time
r = np.array([sum_x(a) for a in A])
r[:5]

Wall time: 1.33 s


array([1830, 1485,   66, 3655,  190])

In [24]:
@np.vectorize
def sum_x_v(x):
    return sum(range(x))

In [25]:
%%time
sum_x_v(A)

Wall time: 8.38 s


array([ 52975, 140715,   6216, ..., 421821, 353220,   4851])

In [26]:
@numba.vectorize
def sum_x_v_nb(x):
    return sum(range(x))

In [29]:
%%time
sum_x_v_nb(A)

<ipython-input-26-38b161574d63>:1: NumbaWarning: 
Compilation is falling back to object mode WITHOUT looplifting enabled because Function "sum_x_v_nb" failed type inference due to: Untyped global name 'sum': Cannot determine Numba type of <class 'builtin_function_or_method'>

File "<ipython-input-26-38b161574d63>", line 3:
def sum_x_v_nb(x):
    return sum(range(x))
    ^

  @numba.vectorize


TypeError: return type must be specified for object mode

3. Приведите все слова из столбца key к верхнему регистру

In [31]:
import pandas as pd
import string
import numpy as np

def create_df(allow_nan=False, N=2_000_000):
    df = pd.DataFrame(np.random.randint(0, 10, (N, 4)), columns=[f"col{i}" for i in range(4)])
    names = ["Apple",  "Banana",  "Apricot",  "Atemoya",  "Avocados",  "Blueberry",  "Blackcurrant",  "Ackee",  "Cranberry",  "Cantaloupe",  "Cherry",  "Black sapote/Chocolate pudding fruit",  "Dragonrfruit",  "Dates",  "Cherimoya",  "Buddha’s hand fruit",  "Finger Lime",  "Fig",  "Coconut",]
    if allow_nan:
        names.append(None)
    df["key"] = np.random.choice(names, N, replace=True)
    return df

In [32]:
df = create_df(allow_nan=False, N=2_000_000)

In [33]:
def f(row):
    if row["col3"] % 2 == 0:
        return row["col0"] - row["col1"]
    return row["col0"] - row["col2"]

In [34]:
%%time
df.apply(
    f,
    axis=1
)

Wall time: 25.4 s


0         -2
1         -2
2         -2
3          4
4          0
          ..
1999995    1
1999996   -5
1999997   -2
1999998   -7
1999999   -3
Length: 2000000, dtype: int64

In [35]:
%%time
np.where(
    df["col3"] % 2 == 0,
    df["col0"] - df["col1"],
    df["col0"] - df["col2"]
)

Wall time: 50 ms


array([-2, -2, -2, ..., -2, -7, -3], dtype=int32)

4\. Для каждой строки рассчитайте разность между значениями col0 и col1, если в столбце col3 стоит четное число, и разность между col0 и col2 в противном случае.

In [ ]:
df = create_df(allow_nan=False, N=500_000).select_dtypes('number')

## Лабораторная работа 2

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy` и `pandas`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy` или структур `pandas` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

В файлах `recipes_sample.csv` и `reviews_sample.csv` (__ЛР 2__) находится информация об рецептах блюд и отзывах на эти рецепты соответственно. Загрузите данные из файлов в виде `pd.DataFrame` с названиями `recipes` и `reviews`. Обратите внимание на корректное считывание столбца(ов) с индексами. Приведите столбцы к нужным типам.

In [3]:
recipes = pd.read_csv(r'data\recipes_sample.csv')
reviews = pd.read_csv(r'data\reviews_sample.csv', index_col=0)

In [4]:
recipes['submitted'] = pd.to_datetime(recipes['submitted'])
reviews['date'] = pd.to_datetime(reviews['date'])

In [5]:
recipes.head()

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN


In [6]:
reviews.head()

,user_id,recipe_id,date,rating,review
370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...


## Numba

В файле `rating_predictions.json` хранятся данные о рейтингах рецептов и прогнозных значениях рейтингов для этого рецепта, полученных при помощи модели машинного обучения. 

Напишите несколько версий функции (см. [MAPE](https://en.wikipedia.org/wiki/Mean_absolute_percentage_error)) для расчета среднего абсолютного процентного отклонения значения рейтинга отзыва на рецепт от прогнозного значения рейтинга для данного рецепта. 


Замечание 1: в формуле MAPE под $A_t$ понимается рейтинг из отзыва $t$, под $F_t$ - прогнозное значения рейтинга отзыва $t$.

Замечание 2: в результате работы функций должно получиться одно число - MAPE для всего набора данных.

№1\.1 Создайте два списка `A_list` и `F_list` на основе файла `rating_predictions.json`. Напишите функцию `mape_lists` без использования векторизованных операций и методов массивов `numpy` и без использования `numba` (проитерируйтесь по спискам и вычислите суммарное значение MAPE для всех элементов, а потом усредните результат).

Измерьте время выполнения данной функции на входных данных `A_list` и `F_list`. Временем, затрачиваемым на создание списков, можно пренебречь.
    

In [7]:
with open(r'data/rating_predictions.json') as f:
    data = json.load(f)
df = pd.DataFrame(data)
df

,rating,prediction
0,5.0,4.944444
1,5.0,4.437500
2,5.0,4.727273
3,5.0,4.354545
4,5.0,4.888889
...,...,...
119886,5.0,4.903226
119887,5.0,4.333333
119888,5.0,5.000000
119889,5.0,4.142857


In [8]:
A_list = df['rating'].to_list()
F_list = df['prediction'].to_list()

In [20]:
def mape_list(lst1, lst2):
    sm = 0
    for t in range(len(lst1)):
        sm += abs((lst1[t] - lst2[t])/lst1[t])
    mape = sm*100/len(lst1)
    return mape

In [23]:
%%time
mape_list(A_list, F_list)

Wall time: 45.9 ms


13.325265503992638

№1\.2. Создайте массивы `numpy` `A_array` и `F_array` на основе списков `A_list` и `F_list`. Напишите функцию `mape_numpy` с использованием векторизованных операций и методов массивов `numpy`.

Измерьте время выполнения данной функции на входных данных `A_array` и `F_array`. Временем, затрачиваемым на создание массивов, можно пренебречь.

In [103]:
A_array = np.array(A_list)
F_array = np.array(F_list)

In [104]:
def mape_numpy(arr1, arr2):
    mape = abs((arr1 - arr2)/arr1).mean()*100
    return mape

In [106]:
%%time
mape_numpy(A_array, F_array)

Wall time: 998 µs


13.32526550399145

№1\.3. Создайте объекты `numba.typed.List` `A_typed` и `F_typed` на основе списков `A_list` и `F_list`. Напишите функцию `mape_numba` без использования векторизованных операций и методов массивов `numpy`, но с использованием `numba`. 

Измерьте время выполнения данной функции на входных данных `A_typed` и `F_typed`. Временем, затрачиваемым на создание объектов `numba.typed.List`, можно пренебречь.

Измерьте время выполнения данной функции на входных данных `A_array` и `F_array`.

In [107]:
A_typed = numba.typed.List(A_list)
F_typed = numba.typed.List(F_list)

In [109]:
@numba.njit
def mape_numba(nmb1, nmb2):
    sm = 0
    for t in range(len(nmb1)):
        sm += abs((nmb1[t] - nmb2[t])/nmb1[t])
    mape = sm/len(nmb1)*100
    return mape

In [119]:
%%time
mape_numba(A_typed, F_typed)

Wall time: 6.98 ms


13.325265503992636

In [118]:
%%time
mape_numba(A_array, F_array)

Wall time: 998 µs


13.325265503992636

## Векторизация

Сайт-агрегатор устроил акцию: он дарит купоны на посещение ресторана тем пользователям, оставившим отзывы, идентификатор которых является _красивым числом_. Натуральное число называется _красивым_, если первая цифра числа совпадает с последней цифрой числа. 



№2\.1 Напишите функцию `is_pretty`, которая для каждого идентификатора пользователя из файла определяет, получит ли он подарок. Запрещается преобразовывать идентификатор пользователя к строке. Подтвердите корректность реализации, продемонстрировав примеры.

In [104]:
ids = reviews["user_id"].unique()
ids

array([     21752,     431813,     400708, ...,    1270706,    2282344,
       2000242659], dtype=int64)

In [118]:
def is_pretty(n: int) -> bool:
    last = n % 10
    while n > 0:
        first = n % 10
        n //= 10
    return first == last

In [120]:
is_pretty(23245)

False

In [121]:
is_pretty(12341)

True

№2\.2 Посчитайте с помощью функции `is_pretty` количество пользователей, которые получат подарок. Выведите это количество на экран. Измерьте время расчетов для входных данных `ids`.

In [135]:
%%timeit
sum([is_pretty(i) for i in ids])

457 ms ± 74.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [131]:
sum([is_pretty(i) for i in ids])

4389

№2\.3. При помощи декоратора `numpy.vectorize` создайте векторизованную версию функции `is_pretty`. Корректно использовав эту векторизованную функцию, посчитайте количество пользователей, которые получат подарок. Выведите это количество на экран. Измерьте время расчетов для входных данных `ids`.


In [144]:
@np.vectorize
def is_pretty1(n: int) -> bool:
    last = n % 10
    while n > 0:
        first = n % 10
        n //= 10
    return first == last

In [146]:
is_pretty1(ids).sum()

4389

In [147]:
%%timeit
is_pretty1(ids).sum()

66.2 ms ± 5.56 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


№2\.4. При помощи декоратора `numba.vectorize` создайте векторизованную версию функции `is_pretty`. Корректно использовав эту векторизованную функцию, посчитайте количество пользователей, которые получат подарок. Выведите это количество на экран. Измерьте время расчетов для входных данных `ids`.


In [149]:
@numba.vectorize
def is_pretty2(n: int) -> bool:
    last = n % 10
    while n > 0:
        first = n % 10
        n //= 10
    return first == last

In [150]:
is_pretty2(ids).sum()

4389

In [151]:
%%timeit
is_pretty2(ids).sum()

976 µs ± 107 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [152]:
is_pretty2(ids).sum()

4389